# PareDiff — Kaggle Training Notebook

**Settings to verify (top-right ⋮):**
- Accelerator: **GPU P100** (16 GB)
- Internet: **ON**
- Persistence: **Variables and files**

**Secrets to attach (Add-ons → Secrets):**
- `WANDB_KEY` — your wandb.ai API key
- `HF_TOKEN` — your Hugging Face write token
- `HF_USER` — your Hugging Face username (e.g. `raipancham2004`)

Just click **Run All** at the top. Come back in ~12 hours.

## 1. Verify GPU is available

In [ ]:
!nvidia-smi

## 2. Install dependencies (~5 min on first run)

In [ ]:
!pip install -q torch torch-geometric diffusers transformers accelerate
!pip install -q torch-scatter torch-sparse -f https://data.pyg.org/whl/torch-2.4.0+cu121.html
!pip install -q wandb huggingface_hub einops

## 3. Authenticate wandb + Hugging Face (uses Kaggle Secrets)

In [ ]:
import os
from kaggle_secrets import UserSecretsClient
secrets = UserSecretsClient()

os.environ['WANDB_API_KEY'] = secrets.get_secret('WANDB_KEY')
os.environ['HF_TOKEN']      = secrets.get_secret('HF_TOKEN')
os.environ['HF_USER']       = secrets.get_secret('HF_USER')

import wandb; wandb.login(key=os.environ['WANDB_API_KEY'])
from huggingface_hub import login as hf_login
hf_login(token=os.environ['HF_TOKEN'])
print('✓ Authenticated wandb + HF')

## 4. Clone PareDiff repo from GitHub

In [ ]:
GITHUB_USER = 'raipancham2004-svg'
REPO        = 'parediff'

!cd /kaggle/working && rm -rf {REPO} && git clone https://github.com/{GITHUB_USER}/{REPO}.git
%cd /kaggle/working/{REPO}
!ls

## 5. Download ISPD'15 dataset (~5 min, cached after first run)

In [ ]:
import os
if not os.path.exists('/kaggle/working/data/ispd15'):
    !mkdir -p /kaggle/working/data
    !cd /kaggle/working/data && git clone --depth 1 https://github.com/The-OpenROAD-Project/OpenROAD-flow-scripts.git ofs
    !mkdir -p /kaggle/working/data/ispd15
    print('✓ Dataset bootstrap complete')
else:
    print('Cached.')

## 6. Resume from previous checkpoint (if exists on HF Hub)

In [ ]:
from huggingface_hub import snapshot_download
HF_REPO = f'{os.environ["HF_USER"]}/parediff-checkpoints'

try:
    snapshot_download(
        repo_id=HF_REPO,
        local_dir='/kaggle/working/parediff/experiments/checkpoints',
        repo_type='model',
    )
    print(f'✓ Resumed from {HF_REPO}')
except Exception as e:
    print(f'No prior checkpoints found (first run is OK): {e}')

## 7. Run smoke test (verifies code works on this GPU)

In [ ]:
%cd /kaggle/working/parediff
!python -m code.test_smoke

## 8. Train PareDiff

Auto-checkpoints every 20 epochs to `/kaggle/working/parediff/experiments/checkpoints/`.

On Kaggle P100 (16 GB), expect:
- ~30 epochs/hour
- Total target: 200 epochs → ~6-7 hours per session
- Convergence: usually 2-3 sessions of 12 hr each = 1-2 days wall clock

In [ ]:
!python -m code.train \
    --data /kaggle/working/data/ispd15 \
    --suite ispd15 \
    --epochs 200 \
    --lr 1e-4 \
    --gnn_hidden 128 \
    --denoiser_hidden 256 \
    --ckpt_dir /kaggle/working/parediff/experiments/checkpoints \
    --use_wandb

## 9. Push checkpoints to Hugging Face Hub (so next session can resume)

In [ ]:
from huggingface_hub import HfApi, create_repo
api = HfApi()
HF_REPO = f'{os.environ["HF_USER"]}/parediff-checkpoints'

try:
    create_repo(HF_REPO, repo_type='model', exist_ok=True, private=True)
    api.upload_folder(
        folder_path='/kaggle/working/parediff/experiments/checkpoints',
        repo_id=HF_REPO,
        repo_type='model',
    )
    print(f'✓ Pushed checkpoints to {HF_REPO}')
except Exception as e:
    print(f'Push failed: {e}')

---

## What to do when this session ends

1. Kaggle session expires after 12 hr → notebook auto-saves
2. Open the notebook again, click **Run All**
3. Cells 1-7 are fast (cached install + dataset)
4. Cell 6 will RESUME from your HF checkpoint
5. Cell 8 continues training from where you left off

**To monitor live:** open your wandb dashboard at https://wandb.ai/<your-username>/parediff and share the URL with Claude in chat.